# OHLCV finalization study — 0.5 s sweep (GeckoTerminal, 1-minute candles)

**What it measures.** For the minute that has just ended, how far each OHLCV value (open, high, low,
close, volume) still is from its true value when you read it 1.0, 1.5, 2.0 … 40.0 seconds after
the minute ended.

**How.** Every minute it makes 3 calls at 3 consecutive offsets of the sweep, and the next minute
continues where it stopped:

| minute | reads at second |
|---|---|
| 1 | 1.0, 1.5, 2.0 |
| 2 | 2.5, 3.0, 3.5 |
| … | … |
| 27 | 40.0 (then the sweep starts again at 1.0) |

Each call returns the last 1000 candles, so every read of a minute can be checked later. **Five
minutes later**, the burst of that minute reads the same candle again: that is taken as its **true
(final) value**. Each early read is then compared with it, field by field. All 3 reads of the
reference burst should agree; `ref_consistent` in the CSV records whether they did (a check on the
5-minute assumption).

**Output:** `<DATA_DIR>/finalization_<COIN_LABEL>.csv`, one row per read: the values read, the true
values, the % change of every field, and whether each field and the whole candle were already
final. Rows are written (and flushed to disk) as soon as their reference burst is in. Reads still
waiting for it are kept in `finalization_<COIN_LABEL>.pending.json`, so an interruption loses
nothing: after a restart they are completed by the next burst (each call still returns the last
~16 hours). Another coin gets its own CSV; the same coin continues its CSV.

**Clock.** Reads are timed with this computer's clock, so it must match the server's. Every call
brackets the server time (its `Date` header) between the send and receive moments; intersecting
these brackets gives the clock offset to within a few tenths of a second. The run **refuses to start** when the clock is
off by more than `MAX_CLOCK_OFFSET_S`; later in a run it warns, and every row stores the offset (the
statistics leave out rows taken with a wrong clock). On Windows: `w32tm /resync` (admin prompt).

**Rate limits.** Every 429 is logged to `<DATA_DIR>/rate_limits.csv` (which call of the burst, its
offset, time since the previous call, calls in the preceding 60 s / 10 s), so you can see whether the
burst or the per-minute total trips the limit.

**Run** the cells; the run cell keeps going until you interrupt the kernel (■). Keep the trading bots
(and anything else using GeckoTerminal) off this IP while it runs: the limit is per IP.

The last cell (**statistics**) is standalone: run it any time, also from another notebook.

## 1. Settings

In [ ]:
POOL_ADDRESS = "9SddWzYaq6KQD4RhXU8wmFHr3BhkX6GjcyF9f4T4vsUB"   # GeckoTerminal pool address (Solana); pick an ACTIVE coin
COIN_LABEL = "FRUNK"                       # CSV: finalization_<COIN_LABEL>.csv (one per coin)
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'live_pipeline').is_dir() and (p / 'development_notebooks').is_dir())   # the repository
DATA_DIR = str(REPO / "research_development" / "finalization_data")   # this study's outputs (not committed)

# The sweep: read the just-ended minute at these seconds after it ended, 3 per minute, in order.
OFFSET_START_S = 1.0
OFFSET_END_S = 40.0
OFFSET_STEP_S = 0.5
REFERENCE_AFTER_S = 300                   # the burst >= 5 min later gives the true values

# API budget (per IP): at most CALLS_PER_MINUTE calls in any rolling 60 s (+ margin). Moving 3 x
# OFFSET_STEP_S = 1.5 s per minute keeps every 4-call span at 61.5 s, just above 60 + margin.
CALLS_PER_MINUTE = 3
SAFETY_MARGIN_S = 1.0
BACKOFF_AFTER_429_S = 180.0

# Clock: refuse to start (first 10 minutes) / warn later when this computer's clock is off by more
# than this, compared with the server's clock.
MAX_CLOCK_OFFSET_S = 1.0

API_BASE = "https://api.geckoterminal.com/api/v2"
NETWORK = "solana"
REQUEST_PARAMS = {"aggregate": "1", "limit": 1000, "currency": "token", "token": "base",
                  "include_empty_intervals": "true"}         # the same request as the bots
REQUEST_TIMEOUT_S = 10.0
PROGRESS_EVERY_MIN = 15                   # print a progress line this often

## 2. Collector

In [ ]:
import csv
import json
import math
import os
import threading
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path

import numpy as np
import requests

FIELDS = ["open", "high", "low", "close", "volume"]
COLUMNS = (["minute_utc", "minute_ts", "offset_s", "age_s", "age_true_s", "clock_offset_s", "clock_offset_pm_s",
            "latency_s", "t_send", "present"]
           + [f"read_{f}" for f in FIELDS] + [f"true_{f}" for f in FIELDS]
           + ["ref_age_s", "ref_consistent", "traded"]
           + [f"pct_{f}" for f in FIELDS] + [f"final_{f}" for f in FIELDS] + ["final_candle"])
OFFSETS = [round(float(x), 3) for x in np.arange(OFFSET_START_S, OFFSET_END_S + OFFSET_STEP_S / 2, OFFSET_STEP_S)]
CHUNKS = [OFFSETS[i:i + CALLS_PER_MINUTE] for i in range(0, len(OFFSETS), CALLS_PER_MINUTE)]


def utc(t):
    return datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


class RollingLimiter:
    """At most n calls in any rolling 60 s (+ margin)."""

    def __init__(self, n, margin):
        self.n, self.margin = n, margin
        self.times = deque(maxlen=64)

    def feasible(self, planned):
        seq = sorted(self.times)[-self.n:] + list(planned)
        start = len(seq) - len(planned)
        return all(seq[k] >= seq[k - self.n] + 60.0 + self.margin
                   for k in range(max(start, self.n), len(seq)))

    def record(self, t):
        self.times.append(t)


_RECENT = None


def _recent_path():
    return Path(DATA_DIR) / "recent_calls.json"


def remember_call(t):
    """Send times of the latest calls, shared by every coin (the budget is per IP)."""
    global _RECENT
    _RECENT = (_RECENT or [])[-7:] + [t]
    try:
        _recent_path().write_text(json.dumps({"coin": COIN_LABEL, "times": _RECENT}))
    except Exception:
        pass


class ClockOffError(RuntimeError):
    pass


class ClockEstimator:
    """Offset of the server clock vs this computer (server - local), from the Date headers.

    The Date header has whole seconds: the server made the response at server time S in [D, D+1)
    while the local clock read between t_send and t_recv, so the offset lies in
    [D - t_recv, D + 1 - t_send]. Calls land at different fractions of a second, so intersecting the
    recent brackets narrows it to about the latency. If the brackets stop overlapping (the clock was
    corrected or drifted), the oldest are dropped."""

    def __init__(self, keep=40):
        self.iv = deque(maxlen=keep)

    def add(self, headers, t_send, t_recv):
        d = next((v for k, v in (headers or {}).items() if k.lower() == "date"), None)
        if not d:
            return
        try:
            D = parsedate_to_datetime(d).timestamp()
        except Exception:
            return
        self.iv.append((D - t_recv, D + 1.0 - t_send))
        while len(self.iv) > 1 and max(l for l, h in self.iv) > min(h for l, h in self.iv):
            self.iv.popleft()

    def bounds(self):
        if not self.iv:
            return None, None
        return max(l for l, h in self.iv), min(h for l, h in self.iv)

    def estimate(self):
        lo, hi = self.bounds()
        if lo is None:
            return None, None
        return (lo + hi) / 2, (hi - lo) / 2


def log_rate_limit(row):
    """Append one 429 event to <DATA_DIR>/rate_limits.csv (shared by all coins: the limit is per IP)."""
    p = Path(DATA_DIR) / "rate_limits.csv"
    cols = ["time_utc", "t_send", "coin", "call_in_burst", "calls_in_burst", "offset_s", "since_prev_call_s",
            "calls_prev_60s", "calls_prev_10s", "other_calls_in_burst_ok"]
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if new:
            w.writeheader()
        w.writerow(row)


def sleep_until(t):
    while True:
        d = t - time.time()
        if d <= 0:
            return
        if d > 0.03:
            time.sleep(min(d - 0.02, 0.5))


_tls = threading.local()


def fetch_ohlcv():
    """One OHLCV call -> (t_send, t_recv, status, headers, body_text, error)."""
    s = getattr(_tls, "session", None)
    if s is None:
        s = _tls.session = requests.Session()
        s.headers["Accept"] = "application/json"
    url = f"{API_BASE}/networks/{NETWORK}/pools/{POOL_ADDRESS}/ohlcv/minute"
    t_send = time.time()
    try:
        r = s.get(url, params=REQUEST_PARAMS, timeout=REQUEST_TIMEOUT_S)
        return t_send, time.time(), r.status_code, dict(r.headers), r.text, None
    except Exception as e:
        return t_send, time.time(), None, {}, None, f"{type(e).__name__}: {e}"


FETCH = fetch_ohlcv


def candles_by_ts(body_text):
    """{minute_ts: (o, h, l, c, v)} of one response."""
    lst = json.loads(body_text)["data"]["attributes"]["ohlcv_list"]
    return {int(r[0]): tuple(float(x) for x in r[1:6]) for r in lst}


class SweepCollector:
    def __init__(self):
        self.path = Path(DATA_DIR) / f"finalization_{COIN_LABEL}.csv"
        self.pending_path = Path(DATA_DIR) / f"finalization_{COIN_LABEL}.pending.json"
        self.path.parent.mkdir(parents=True, exist_ok=True)
        global _RECENT
        try:
            rec = json.loads(_recent_path().read_text())
            _RECENT = rec.get("times", [])
            busy = time.time() - _recent_path().stat().st_mtime < 120
            if busy and rec.get("coin") != COIN_LABEL:
                raise RuntimeError(f"coin '{rec.get('coin')}' is still being collected (last call "
                                   f"{time.time() - _recent_path().stat().st_mtime:.0f}s ago); one coin at a time")
        except (FileNotFoundError, json.JSONDecodeError, AttributeError):
            _RECENT = []
        self.limiter = RollingLimiter(CALLS_PER_MINUTE, SAFETY_MARGIN_S)
        for t in _RECENT:
            self.limiter.record(t)
        self.chunk_idx = 0
        self.pending = []           # reads waiting for their reference burst
        self.stats = {"calls": 0, "ok": 0, "429": 0, "failed": 0, "rows": 0, "not_present": 0,
                      "expired": 0, "sweeps": 0}
        self.pause_until = 0.0
        self.first_ok = True
        self.clock = ClockEstimator()
        self.started = time.time()
        self.next_clock_warning = 0.0
        self.limited_by_position = {}
        self._repair_csv()
        if self.path.exists() and self.path.stat().st_size > 0:
            with open(self.path, newline="") as f:
                header = next(csv.reader(f), [])
            if header != COLUMNS:
                raise RuntimeError(f"{self.path} was written by an older version of this notebook (different "
                                   f"columns). Move or delete it (and its .pending.json), then run again.")
        if self.pending_path.exists():
            st = json.loads(self.pending_path.read_text())
            if st.get("pool") not in (None, POOL_ADDRESS):
                raise RuntimeError(f"{self.pending_path} belongs to pool {st['pool']}; use another COIN_LABEL")
            self.chunk_idx = int(st.get("chunk_idx", 0)) % len(CHUNKS)
            self.pending = st.get("pending", [])
            print(f"continuing {self.path.name}: next offsets {CHUNKS[self.chunk_idx]}, "
                  f"{len(self.pending)} reads waiting for their reference")
        new = not self.path.exists() or self.path.stat().st_size == 0
        self.file = open(self.path, "a", newline="")
        self.writer = csv.DictWriter(self.file, fieldnames=COLUMNS)
        if new:
            self.writer.writeheader()
            self._sync()

    def _repair_csv(self):
        """A hard stop can leave a half-written last line: cut back to the last full one."""
        if not self.path.exists() or self.path.stat().st_size == 0:
            return
        with open(self.path, "rb+") as f:
            size = f.seek(0, os.SEEK_END)
            f.seek(max(0, size - 65536))
            tail = f.read()
            if not tail.endswith(b"\n"):
                keep = size - len(tail) + tail.rfind(b"\n") + 1 if b"\n" in tail else 0
                f.truncate(keep)
                print(f"removed a partial last line ({size - keep} bytes) left by an interruption")

    def _sync(self):
        self.file.flush()
        os.fsync(self.file.fileno())

    def _save_pending(self):
        tmp = self.pending_path.with_suffix(".tmp")
        tmp.write_text(json.dumps({"pool": POOL_ADDRESS, "chunk_idx": self.chunk_idx, "pending": self.pending}))
        os.replace(tmp, self.pending_path)                      # atomic

    # ------------------------------------------------------------------ one burst
    def process_burst(self, m_start, results):
        """results: [(offset, (t_send, t_recv, status, headers, body, err)), ...] in send order.
        Store the reads of the minute that just ended; resolve reads >= 5 min old with this burst."""
        target = int(m_start) - 60                              # the minute that just ended
        burst = []                                              # (t_mid, candles) of the ok calls
        for off, (t_send, t_recv, status, headers, body, err) in results:
            if status == 200:
                self.clock.add(headers, t_send, t_recv)
        self._check_clock()
        clk, clk_pm = self.clock.estimate()
        n_ok = sum(1 for _, r in results if r[2] == 200)
        for pos, (off, (t_send, t_recv, status, headers, body, err)) in enumerate(results, start=1):
            self.stats["calls"] += 1
            if status == 200 and body:
                try:
                    candles = candles_by_ts(body)
                except Exception as e:
                    self.stats["failed"] += 1
                    print(f"{utc(t_send)} unreadable response: {e}")
                    continue
                self.stats["ok"] += 1
                t_mid = (t_send + t_recv) / 2
                burst.append((t_send, t_mid, candles))
                val = candles.get(target)
                if val is None:
                    self.stats["not_present"] += 1
                self.pending.append({"minute_ts": target, "offset_s": off, "t_send": t_send,
                                     "age_s": t_mid - (target + 60), "latency_s": t_recv - t_send,
                                     "clock_offset_s": clk, "clock_offset_pm_s": clk_pm,
                                     "read": list(val) if val else None})
                if self.first_ok:
                    self.first_ok = False
                    vols = [v[4] for k, v in sorted(candles.items())[-60:]]
                    print(f"first response: {len(candles)} candles, traded minutes in the last hour "
                          f"{np.mean([v > 0 for v in vols]):.0%}")
                    if np.mean([v > 0 for v in vols]) < 0.3:
                        print("WARNING: few traded minutes; an inactive coin gives few informative reads")
                    if clk is not None:
                        print(f"clock: server - local = {clk:+.2f} s (±{clk_pm:.2f} s; narrows over the next calls)")
            elif status == 429:
                self.stats["429"] += 1
                self.pause_until = time.time() + BACKOFF_AFTER_429_S
                prev = sorted(t for t in self.limiter.times if t < t_send - 0.05)   # not this call itself
                since = t_send - prev[-1] if prev else None
                n60 = sum(1 for t in prev if t >= t_send - 60.0)
                n10 = sum(1 for t in prev if t >= t_send - 10.0)
                key = f"{pos} of {len(results)}"
                self.limited_by_position[key] = self.limited_by_position.get(key, 0) + 1
                log_rate_limit({"time_utc": utc(t_send), "t_send": round(t_send, 3), "coin": COIN_LABEL,
                                "call_in_burst": pos, "calls_in_burst": len(results), "offset_s": off,
                                "since_prev_call_s": round(since, 3) if since is not None else None,
                                "calls_prev_60s": n60, "calls_prev_10s": n10, "other_calls_in_burst_ok": n_ok})
                print(f"{utc(t_send)} RATE LIMITED (429) on call {pos} of {len(results)} (second {off}); "
                      f"previous call {since:.1f}s earlier; {n60} call(s) in the preceding 60 s, {n10} in 10 s; "
                      f"{n_ok} other call(s) of this burst went through; pausing {BACKOFF_AFTER_429_S:.0f}s"
                      if since is not None else
                      f"{utc(t_send)} RATE LIMITED (429) on call {pos} of {len(results)} (second {off}); "
                      f"pausing {BACKOFF_AFTER_429_S:.0f}s")
            else:
                self.stats["failed"] += 1
                print(f"{utc(t_send)} call failed: status {status} {err or ''}")

        # resolve every read whose minute ended >= REFERENCE_AFTER_S before this burst
        if burst:
            first_send = burst[0][0]
            for p in list(self.pending):
                if p["minute_ts"] + 60 + REFERENCE_AFTER_S > first_send:
                    continue
                refs = [c.get(p["minute_ts"]) for _, _, c in burst]
                refs = [r for r in refs if r is not None]
                if refs:
                    self._write_row(p, refs, burst[-1][1])
                else:                                            # out of the 1000-candle window
                    self.stats["expired"] += 1
                self.pending.remove(p)
            self._sync()

    def _check_clock(self):
        """Refuse to run (first 10 minutes) or warn (later) when the clock is surely off by more
        than MAX_CLOCK_OFFSET_S."""
        lo, hi = self.clock.bounds()
        if lo is None or not (lo > MAX_CLOCK_OFFSET_S or hi < -MAX_CLOCK_OFFSET_S):
            return
        est = (lo + hi) / 2
        msg = (f"this computer's clock is {abs(est):.1f} s {'ahead of' if est < 0 else 'behind'} the server's "
               f"(server - local between {lo:+.2f} and {hi:+.2f} s), more than MAX_CLOCK_OFFSET_S = "
               f"{MAX_CLOCK_OFFSET_S} s")
        if time.time() - self.started < 600:
            raise ClockOffError(msg)
        if time.time() >= self.next_clock_warning:
            self.next_clock_warning = time.time() + 3600
            print(f"{utc(time.time())} WARNING: {msg}. Rows keep running but are marked with the offset "
                  f"(the statistics leave them out). Re-sync the clock: w32tm /resync (admin prompt).")

    def _write_row(self, p, refs, ref_t_mid):
        true = refs[-1]
        read = p["read"]
        clk = p.get("clock_offset_s")
        row = {"minute_utc": utc(p["minute_ts"]), "minute_ts": p["minute_ts"], "offset_s": p["offset_s"],
               "age_s": round(p["age_s"], 4),
               "age_true_s": round(p["age_s"] + clk, 4) if clk is not None else None,
               "clock_offset_s": round(clk, 4) if clk is not None else None,
               "clock_offset_pm_s": round(p["clock_offset_pm_s"], 4) if clk is not None else None,
               "latency_s": round(p["latency_s"], 4),
               "t_send": round(p["t_send"], 4), "present": read is not None,
               "ref_age_s": round(ref_t_mid - (p["minute_ts"] + 60), 2),
               "ref_consistent": all(r == true for r in refs), "traded": true[4] > 0}
        all_final = read is not None
        for i, f in enumerate(FIELDS):
            row[f"true_{f}"] = true[i]
            if read is None:
                row[f"final_{f}"] = False
                continue
            x, y = read[i], true[i]
            row[f"read_{f}"] = x
            row[f"pct_{f}"] = (x - y) / y * 100.0 if y != 0 else (0.0 if x == 0 else None)
            row[f"final_{f}"] = x == y
            all_final &= x == y
        row["final_candle"] = all_final
        self.writer.writerow(row)
        self.stats["rows"] += 1

    # ------------------------------------------------------------------ main loop
    def run_forever(self):
        print(f"collecting {COIN_LABEL} ({POOL_ADDRESS}) into {self.path}: offsets {OFFSETS[0]}..{OFFSETS[-1]} s "
              f"every {OFFSET_STEP_S} s, {len(CHUNKS)} minutes per sweep — interrupt the kernel to stop")
        pool = ThreadPoolExecutor(max_workers=CALLS_PER_MINUTE)
        next_progress = time.time() + PROGRESS_EVERY_MIN * 60
        inflight, m_start = [], None
        try:
            while True:
                try:
                    if time.time() < self.pause_until:
                        time.sleep(min(5.0, self.pause_until - time.time()))
                        continue
                    offs = CHUNKS[self.chunk_idx]
                    now = time.time()
                    m_start = math.floor(now / 60.0) * 60.0
                    while not (m_start + offs[0] >= now + 0.3 and
                               self.limiter.feasible([m_start + o for o in offs])):
                        m_start += 60.0
                    for o in offs:
                        sleep_until(m_start + o)
                        t = time.time()
                        self.limiter.record(t)
                        remember_call(t)
                        inflight.append((o, pool.submit(FETCH)))
                    results = [(o, f.result()) for o, f in inflight]
                    inflight = []
                    self.process_burst(m_start, results)
                    self.chunk_idx = (self.chunk_idx + 1) % len(CHUNKS)
                    if self.chunk_idx == 0:
                        self.stats["sweeps"] += 1
                    self._save_pending()
                    if time.time() >= next_progress:
                        next_progress += PROGRESS_EVERY_MIN * 60
                        s = self.stats
                        print(f"{utc(time.time())} calls {s['calls']} (ok {s['ok']}, 429 {s['429']}, failed "
                              f"{s['failed']}), rows written {s['rows']}, sweeps {s['sweeps']}, "
                              f"waiting {len(self.pending)}, minute not yet listed {s['not_present']}"
                              + (f", 429 by call position {self.limited_by_position}" if self.limited_by_position else "")
                              + (f", clock {self.clock.estimate()[0]:+.2f} s" if self.clock.iv else ""))
                except (KeyboardInterrupt, ClockOffError):
                    raise
                except Exception as e:
                    print(f"{utc(time.time())} error: {type(e).__name__}: {e}")
                    time.sleep(5.0)
        except KeyboardInterrupt:
            print("interrupted — saving...")
        except ClockOffError as e:
            inflight = []
            self.pending = [p for p in self.pending if p["t_send"] < self.started]   # drop this session's reads
            print(f"NOT STARTED: {e}.\nSync the clock (Windows, admin prompt: w32tm /resync; check on time.is), "
                  f"then run this cell again. The reads of this session were not stored.")
        finally:
            if inflight and m_start is not None:                # calls already sent: keep their reads
                try:
                    done = [(o, f.result(timeout=REQUEST_TIMEOUT_S + 2)) for o, f in inflight]
                    self.process_burst(m_start, done)
                    self.chunk_idx = (self.chunk_idx + 1) % len(CHUNKS)
                except Exception:
                    pass
            self._save_pending()
            self._sync()
            self.file.close()
            pool.shutdown(wait=False)
            print(f"stopped. {self.stats['rows']} rows written this session; {len(self.pending)} reads "
                  f"waiting for their reference are saved and will be completed on the next run.")

## 3. Run (until interrupted)

In [ ]:
collector = SweepCollector()
collector.run_forever()

## 4. Statistics (standalone)

Needs nothing else from the notebook: it reads every `finalization_*.csv` in `STATS_DATA_DIR` and
shows, per coin and across coins:

* per offset (seconds after the minute ended): how many reads, the share whose value was already
  final (equal to the true value 5 minutes later), per field and for the whole candle, with a 95%
  confidence interval, and the size of the remaining change as a percentage of the true value;
* the **earliest second at which 95 / 99 / 99.9 % of candles were final** (from a curve fitted to
  never decrease with time, which pools neighbouring offsets), and the same for "within 0.01 / 0.1 /
  1 % of the true value";
* a check of the 5-minute assumption (`ref_consistent`);
* the clock: rows read while this computer's clock was off by more than `MAX_CLOCK_OFFSET_S` are
  left out (their "second 5" was not really second 5);
* the same charts and table for **all coins together** (every coin's reads pooled per offset), after
  the individual coins, then a summary row per coin;
* the rate-limit log (`rate_limits.csv`): which call of the burst was refused, and how busy the
  preceding minute was.

Each offset is read once per 27-minute sweep (~53 reads per offset per day), so a 99 % figure needs a
few days of collection; the confidence intervals show how settled the numbers are.

In [ ]:
# ===================== settings of this cell =====================
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'live_pipeline').is_dir() and (p / 'development_notebooks').is_dir())   # the repository
STATS_DATA_DIR = str(REPO / "research_development" / "finalization_data")
ONLY_TRADED = True                     # only minutes with trades (untraded minutes rarely change)
TARGETS = (0.95, 0.99, 0.999)          # share of candles final
TOLERANCES_PCT = (0.01, 0.1, 1.0)      # "close enough": every field within this % of the true value
MAX_CLOCK_OFFSET_S = 1.0               # leave out rows read while the clock was off by more than this
SHOW_PLOTS = True
# =================================================================
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter
from IPython.display import display, Markdown

# The dark theme of the other notebooks' charts (profitability_visuals, strategy)
THEME = {
    'figure': '#0f172a', 'plot': '#0a0e1a', 'text': '#e2e8f0', 'muted': '#94a3b8', 'grid': '#334155',
    'candle': '#e2e8f0', 'close': '#f59e0b', 'high': '#60a5fa', 'low': '#ef4444', 'volume': '#10b981',
}
RC = {
    'figure.facecolor': THEME['figure'], 'axes.facecolor': THEME['plot'], 'savefig.facecolor': THEME['figure'],
    'axes.edgecolor': THEME['grid'], 'axes.labelcolor': THEME['text'], 'text.color': THEME['text'],
    'xtick.color': THEME['muted'], 'ytick.color': THEME['muted'], 'grid.color': THEME['grid'],
    'grid.alpha': 0.6, 'axes.grid': True, 'axes.axisbelow': True, 'font.size': 11,
    'legend.facecolor': THEME['figure'], 'legend.edgecolor': THEME['grid'],
}
PCT_FLOOR = 1e-3                       # chart 2: linear below this % (so 0% is drawn at 0), log above

_F = ["open", "high", "low", "close", "volume"]


def _isotonic(y, w):
    """Weighted fit that never decreases with the offset (pool adjacent violators)."""
    out = np.full(len(y), np.nan)
    blocks = []
    for i in np.flatnonzero(w > 0):
        blocks.append([y[i] * w[i], w[i], [i]])
        while len(blocks) > 1 and blocks[-2][0] / blocks[-2][1] > blocks[-1][0] / blocks[-1][1]:
            a, b = blocks.pop(), blocks.pop()
            blocks.append([a[0] + b[0], a[1] + b[1], b[2] + a[2]])
    for s, ww, members in blocks:
        out[members] = s / ww
    return out


def _wilson(k, n, z=1.96):
    with np.errstate(invalid="ignore", divide="ignore"):
        p = k / n
        d = 1 + z * z / n
        c = (p + z * z / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def _first_offset(offsets, fitted, q):
    hit = np.flatnonzero(fitted >= q)
    return float(offsets[hit[0]]) if len(hit) else np.nan


def _bool(s):
    return s.astype(str).str.lower().isin(["true", "1"])


def _load(path):
    df = pd.read_csv(path, on_bad_lines="skip", low_memory=False)
    df = df.drop_duplicates(["minute_ts", "offset_s", "t_send"])
    for c in ["present", "ref_consistent", "traded", "final_candle"] + [f"final_{f}" for f in _F]:
        df[c] = _bool(df[c])
    return df


def _per_offset(d):
    g = d.groupby("offset_s")
    t = pd.DataFrame({"reads": g.size(), "minute_listed": g["present"].mean()})
    for c in ["candle"] + _F:
        k = g[f"final_{c}"].sum()
        t[f"final_{c}"] = k / t["reads"]
        t[f"final_{c}_fit"] = _isotonic(t[f"final_{c}"].to_numpy(), t["reads"].to_numpy().astype(float))
        lo, hi = _wilson(k.to_numpy().astype(float), t["reads"].to_numpy().astype(float))
        t[f"final_{c}_ci_lo"], t[f"final_{c}_ci_hi"] = lo, hi
    absd = d[[f"pct_{f}" for f in _F]].abs()
    within_all = {}
    for tol in TOLERANCES_PCT:
        ok = d["present"] & (absd.fillna(np.inf) <= tol).all(axis=1)
        t[f"within_{tol}%"] = ok.groupby(d["offset_s"]).mean()
        t[f"within_{tol}%_fit"] = _isotonic(t[f"within_{tol}%"].to_numpy(), t["reads"].to_numpy().astype(float))
    for f in ("close", "high", "low", "volume"):
        a = d[f"pct_{f}"].abs()
        t[f"{f}_mean_abs_pct"] = a.groupby(d["offset_s"]).mean()
        t[f"{f}_p99_abs_pct"] = a.groupby(d["offset_s"]).quantile(0.99)
    return t.reset_index()


def _plot(t, label):
    """The two charts: share of reads already final, and the size of the remaining change as a
    percentage of the true value."""
    offs = t["offset_s"].to_numpy()
    with plt.rc_context(RC):
        fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
        for c in ("candle", "close", "volume"):
            col = THEME[c]
            ax[0].fill_between(offs, t[f"final_{c}_ci_lo"], t[f"final_{c}_ci_hi"], color=col, alpha=.12, linewidth=0)
            ax[0].plot(offs, t[f"final_{c}"], ".", color=col, alpha=.6)
            ax[0].plot(offs, t[f"final_{c}_fit"], "-", color=col, label=c, lw=2 if c == "candle" else 1.3)
        for q in TARGETS:
            ax[0].axhline(q, color=THEME["muted"], lw=.7, ls=":")
        ax[0].set_ylim(max(0, t["final_candle"].min() - .05), 1.005)
        ax[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax[0].set_xlabel("seconds after the minute ended")
        ax[0].set_ylabel("share of reads already final")
        ax[0].set_title(f"{label}: share of reads already final\n(dots: measured, line: fitted, band: 95% CI)",
                        loc="left", fontsize=11.5)
        for f in ("close", "high", "low", "volume"):
            ax[1].plot(offs, t[f"{f}_mean_abs_pct"], color=THEME[f], label=f, lw=1.4)
        # % of the true value: linear below PCT_FLOOR so an exact match is drawn at 0 %, log above
        ax[1].set_yscale("symlog", linthresh=PCT_FLOOR, linscale=0.5)
        top = max(1.0, float(np.nanmax(t[[f"{f}_mean_abs_pct" for f in ("close", "high", "low", "volume")]].to_numpy())))
        ticks = [0.0] + [10.0 ** e for e in range(int(np.log10(PCT_FLOOR)), int(np.ceil(np.log10(top))) + 1)]
        ax[1].yaxis.set_major_locator(FixedLocator(ticks))
        ax[1].yaxis.set_minor_locator(FixedLocator([]))
        ax[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: "0%" if v == 0 else f"{v:g}%"))
        ax[1].set_ylim(0, ticks[-1] * 1.5)
        ax[1].set_xlabel("seconds after the minute ended")
        ax[1].set_ylabel("mean |read - true value|, % of the true value")
        ax[1].set_title(f"{label}: size of the remaining change\n(% difference from the true value)",
                        loc="left", fontsize=11.5)
        for a in ax:
            a.legend()
        plt.tight_layout()
        plt.show()


summary = []
pooled = []                            # every coin's analysed reads, for the charts of all coins together
files = sorted(Path(STATS_DATA_DIR).glob("finalization_*.csv"))
if not files:
    print(f"no finalization_*.csv in {STATS_DATA_DIR}")
for path in files:
    coin = path.stem.replace("finalization_", "", 1)
    df = _load(path)
    clock_note = "no clock data (older file)"
    if "clock_offset_s" in df and df["clock_offset_s"].notna().any():
        off = df["clock_offset_s"]
        bad = off.abs() > MAX_CLOCK_OFFSET_S
        clock_note = (f"clock offset (server - local) median {off.median():+.2f} s, range {off.min():+.2f} .. "
                      f"{off.max():+.2f} s; {int(bad.sum())} rows left out (off by > {MAX_CLOCK_OFFSET_S} s)")
        df = df[~bad]
    if not len(df):
        display(Markdown(f"### {coin}\nno rows yet (rows appear 5 minutes after their read)"))
        continue
    d = df[df["traded"]] if ONLY_TRADED else df
    hours = (df["t_send"].max() - df["t_send"].min()) / 3600
    t = _per_offset(d)
    offs = t["offset_s"].to_numpy()
    row = {"coin": coin, "hours": round(hours, 1), "reads": len(d), "minutes": d["minute_ts"].nunique(),
           "min_reads_per_offset": int(t["reads"].min()), "traded_share": df["traded"].mean(),
           "ref_consistent": df["ref_consistent"].mean()}
    for c in ("candle", "close", "volume"):
        for q in TARGETS:
            row[f"{c} {q:.1%} final at (s)"] = _first_offset(offs, t[f"final_{c}_fit"].to_numpy(), q)
    for tol in TOLERANCES_PCT:
        row[f"99% within {tol}% at (s)"] = _first_offset(offs, t[f"within_{tol}%_fit"].to_numpy(), 0.99)
    summary.append(row)

    display(Markdown(
        f"### {coin}\n{len(df)} reads of {df['minute_ts'].nunique()} minutes over {hours:.1f} h; "
        f"{'traded minutes only: ' + str(len(d)) + ' reads; ' if ONLY_TRADED else ''}"
        f"{int(t['reads'].min())}–{int(t['reads'].max())} reads per offset. "
        f"Reference burst consistent (5-minute assumption holds): **{df['ref_consistent'].mean():.2%}**"
        + ("" if df["ref_consistent"].all() else " — some candles still changed within the reference "
           "burst, so 5 minutes was not always final") + f"  \nClock: {clock_note}."))
    first = pd.DataFrame({c: [_first_offset(offs, t[f"final_{c}_fit"].to_numpy(), q) for q in TARGETS]
                          for c in ["candle"] + _F}, index=[f"{q:.1%} final at second" for q in TARGETS])
    display(first)
    show = ["offset_s", "reads", "minute_listed", "final_candle", "final_candle_ci_lo", "final_candle_ci_hi",
            "final_close", "final_volume"] + [f"within_{tol}%" for tol in TOLERANCES_PCT] + \
           ["close_mean_abs_pct", "close_p99_abs_pct", "volume_mean_abs_pct", "volume_p99_abs_pct"]
    display(t[show].round(4))

    if SHOW_PLOTS:
        _plot(t, coin)
    pooled.append(d)

if pooled:
    allc = pd.concat(pooled, ignore_index=True)
    n_coins = len(pooled)
    display(Markdown(f"### All coins together\n{len(allc)} reads of {n_coins} coin(s)"
                     f"{' (traded minutes only)' if ONLY_TRADED else ''}, pooled per offset."))
    t_all = _per_offset(allc)
    offs_all = t_all["offset_s"].to_numpy()
    display(pd.DataFrame({c: [_first_offset(offs_all, t_all[f"final_{c}_fit"].to_numpy(), q) for q in TARGETS]
                          for c in ["candle"] + _F}, index=[f"{q:.1%} final at second" for q in TARGETS]))
    if SHOW_PLOTS:
        _plot(t_all, "All coins")

if summary:
    display(Markdown("### Summary per coin"))
    display(pd.DataFrame(summary).set_index("coin").round(3))

_rl = Path(STATS_DATA_DIR) / "rate_limits.csv"
if _rl.exists():
    rl = pd.read_csv(_rl, on_bad_lines="skip")
    display(Markdown(f"### Rate limits (429): {len(rl)} events"))
    if len(rl):
        by = rl.groupby(["coin", "call_in_burst"]).agg(
            events=("t_send", "size"), median_since_prev_call_s=("since_prev_call_s", "median"),
            median_calls_prev_60s=("calls_prev_60s", "median"), max_calls_prev_60s=("calls_prev_60s", "max"),
            other_calls_in_burst_ok=("other_calls_in_burst_ok", "mean"))
        display(by.round(2))
        print("Mostly call 2-3 of a burst -> the calls close together trip it (spread the burst);\n"
              "mostly call 1 after ~60 s quiet -> the per-minute total or other traffic on this IP.")